# Solución — Actividad Clase 4

Ejemplo de referencia con MSFT. Los grupos pueden usar otra acción y deben escribir su propia interpretación.

**Profesor:** Orlando Joaqui-Barandica  
**Curso:** Introducción a la programación para analítica financiera

In [ ]:
# En Colab, si hace falta:
# !pip -q install yfinance

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TICKER = 'MSFT'
MERCADO = '^GSPC'
INICIO = '2024-10-01'
FIN = '2026-10-01'
RUTA = 'datos/'
os.makedirs('resultados', exist_ok=True)

## 1. Consulta y respaldo

In [ ]:
def cargar_precios(ticker):
    try:
        import yfinance as yf
        bruto = yf.download([ticker, MERCADO], start=INICIO, end=FIN, auto_adjust=True, progress=False)
        if bruto.empty:
            raise ValueError('La descarga llego vacia.')
        precios = bruto['Close'].rename(columns={ticker:'Activo', MERCADO:'Mercado'})[['Activo','Mercado']].dropna()
        return precios, 'yfinance'
    except Exception as error:
        print('No fue posible usar yfinance:', error)
        respaldo = pd.read_csv(RUTA + 'mercado_respaldo_clase4.csv', parse_dates=['Date']).set_index('Date')
        columna = ticker + '_SIM'
        if columna not in respaldo.columns:
            columna = 'MSFT_SIM'
        precios = respaldo[[columna,'MARKET_SIM']].rename(columns={columna:'Activo','MARKET_SIM':'Mercado'})
        return precios.loc[INICIO:FIN].dropna(), 'respaldo_simulado'

precios, fuente = cargar_precios(TICKER)
print('Fuente:', fuente)
display(precios.head())

## 2. Retornos, medias y volatilidad

In [ ]:
retornos = precios.pct_change().dropna()
analisis = pd.DataFrame(index=precios.index)
analisis['precio'] = precios['Activo']
analisis['media_20'] = analisis['precio'].rolling(20).mean()
analisis['media_60'] = analisis['precio'].rolling(60).mean()
analisis['retorno'] = precios['Activo'].pct_change()
analisis['vol_20_anual'] = analisis['retorno'].rolling(20).std() * np.sqrt(252)
display(analisis.tail())

## 3. Estadística, correlación y beta

In [ ]:
display(retornos['Activo'].describe())
retorno_medio = retornos['Activo'].mean()
vol_anual = retornos['Activo'].std() * np.sqrt(252)
correlacion = retornos['Activo'].corr(retornos['Mercado'])
beta, alpha = np.polyfit(retornos['Mercado'], retornos['Activo'], 1)
r2 = correlacion**2

print('Retorno medio diario:', retorno_medio)
print('Volatilidad anualizada:', vol_anual)
print('Correlacion:', correlacion)
print('Beta:', beta)
print('Alpha diario:', alpha)
print('R2:', r2)

## 4. Visualizaciones

In [ ]:
analisis[['precio','media_20','media_60']].plot(figsize=(11,5), title=f'{TICKER}: precio y medias moviles')
plt.show()

In [ ]:
analisis['vol_20_anual'].plot(figsize=(11,4), title=f'{TICKER}: volatilidad movil anualizada')
plt.show()

In [ ]:
x = retornos['Mercado']
y = retornos['Activo']
orden = np.argsort(x.to_numpy())
x_ordenado = x.to_numpy()[orden]
plt.figure(figsize=(8,5))
plt.scatter(x,y,alpha=0.30)
plt.plot(x_ordenado, alpha + beta*x_ordenado)
plt.xlabel('Retorno del mercado')
plt.ylabel(f'Retorno de {TICKER}')
plt.title(f'{TICKER} vs mercado - beta = {beta:.2f}')
plt.show()

## 5. Interpretación de referencia

La interpretación debe mencionar el periodo analizado, la volatilidad, la correlación, el beta y una limitación. No debe convertir una estimación histórica en una predicción.